# Evaluation Agent flow inspector — fixed 10-event case

This notebook uses the exact 10 `user_speech` events supplied in the Debug Panel case. It makes no database write. By default, it reads `backend/.env` and makes four real fine-tuned-model calls: the production baseline plus three prompt/context variants. Each result includes raw model output for direct comparison. The production path also removes model-invented type 2/5 moves when their required `userMessage`/`aiResponse` input is absent.

## Runtime initiation path

- Frontend automatic analysis needs a new completed transcript chunk plus one trigger: 2 seconds of silence, 60 new transcript words, or 45 seconds of continuous speech.
- `runIncrementalPipeline` additionally requires a session with `manipulation_state`, a non-empty cursor tail, and non-empty new speech.
- `runTurnPipeline` starts Evaluation Agent only when the tail has a real user/AI message skeleton or substantive participant content. Otherwise it advances the cursor without initiating the agent.
- A chat Send follows the same `runTurnPipeline` gate, but does not require speech.

For this fixed fixture, the notebook assumes the session-state precondition is already true, as it must be in a normal assigned test session.

## LLM experiments

1. **Production baseline** — current system prompt and current user prompt, through `evaluateTimelinePartition`.
2. **Speech-aware user message** — unchanged system prompt; user message says to combine adjacent fragments and return a content move.
3. **Speech-aware system prompt** — unchanged production user message; system prompt adds the same instruction.
4. **Combined-speech context** — unchanged system prompt; user message supplies the raw events plus one combined transcript.

Set `RUN_LLM_EXPERIMENTS = False` if you only want to inspect prompts and gates without making API calls.

In [ ]:
import json
import subprocess
from pathlib import Path

PROJECT = Path('/Users/yanliu/Projects/Manipulation research-platform')
BACKEND = PROJECT / 'backend'
RUN_LLM_EXPERIMENTS = True  # Set False to inspect gates/prompts without calling the LLM.

# Exact 10-event fixture from the Debug Panel.
TEST_SPEECH = [
    "Okay, so I'm thinking about",
    'Maybe I should consider the target audience',
    'Group age',
    "Um, for this platform, I'm thinking about for",
    'Five to forty years old',
    'And then like considering what is probably the most important',
    "Thing for them. I'm thinking about career advice platform",
    'Yeah, so',
    'I guess first going to have with career advice',
    'Price platform for 25 to 40 years old.',
]

# Keep dist synchronized with the current TypeScript prompt/gate helpers.
subprocess.run(['npm', 'run', 'build'], cwd=BACKEND, check=True)

inspection_script = r'''
const fs = require('fs');
const dotenv = require('dotenv');
const config = dotenv.parse(fs.readFileSync('.env'));
// Explicitly assign parsed values before loading the evaluator, so this
// inspection uses exactly backend/.env without printing the secret.
process.env.OPENAI_API_KEY = config.OPENAI_API_KEY;
process.env.EVAL_MODEL_ID = config.EVAL_MODEL_ID;
const OpenAI = require('openai').default;
const { hasSubstantiveParticipantContent, EVALUATION_SYSTEM_PROMPT, buildEvaluationUserPrompt, evaluateTimelinePartition } = require('./dist/agents/moveTypeEvaluator');
const runExperiments = process.argv[1] === 'true';
const speech = JSON.parse(process.argv[2]);
const partition = speech.map((text, index) => ({
  eventId: index + 1,
  timestampStart: 1000 + index * 1000,
  timestampEnd: 1800 + index * 1000,
  eventType: 'user_speech',
  actionDescription: '',
  speech: text,
  userMessage: '',
  aiResponse: '',
}));
const hasNewSpeech = partition.some((event) => event.eventType === 'user_speech' && event.speech.trim().length > 0);
const hasInteractionSkeleton = partition.some((event) => event.userMessage.trim() || event.aiResponse.trim());
const hasSubstantiveContent = hasSubstantiveParticipantContent(partition);
const runTurnGatePasses = partition.length > 0 && (hasInteractionSkeleton || hasSubstantiveContent);
const automaticGatePasses = runTurnGatePasses && hasNewSpeech;
const combinedSpeech = partition.map((event) => event.speech).join(' ');
const parseJsonArray = (raw) => { try { const value = JSON.parse(raw); return Array.isArray(value) ? value : null; } catch { return null; } };
const callVariant = async (name, systemPrompt, userPrompt) => {
  try {
    const startedAt = Date.now();
    const completion = await new OpenAI({ apiKey: config.OPENAI_API_KEY }).chat.completions.create({
      model: config.EVAL_MODEL_ID,
      messages: [{ role: 'system', content: systemPrompt }, { role: 'user', content: userPrompt }],
      temperature: 0,
      max_tokens: 4096,
    });
    const raw = completion.choices[0]?.message?.content || '';
    return { name, durationMs: Date.now() - startedAt, raw, parsedJsonArray: parseJsonArray(raw), error: null };
  } catch (error) {
    return { name, durationMs: null, raw: null, parsedJsonArray: null, error: error instanceof Error ? error.message : String(error) };
  }
};
(async () => {
const experiments = [];
if (runExperiments) {
  const production = await evaluateTimelinePartition(partition, '', 45_000);
  experiments.push({
    name: '1. production_baseline',
    route: 'evaluateTimelinePartition (production parser/retry path)',
    systemPrompt: EVALUATION_SYSTEM_PROMPT,
    userPrompt: buildEvaluationUserPrompt(partition),
    modelUsed: production.modelUsed, usedFallback: production.usedFallback, retryCount: production.retryCount, durationMs: production.durationMs,
    raw: production.raw, parsedMoves: production.moves,
  });
  const speechAwareUserPrompt = `This is a speech-only design-ideation partition. There are no sent chat messages and no AI responses, so do not create type 2 Ask or type 5 AI_Response moves.\n\nThe events below are adjacent fragments from one continuous participant thought. Combine their meaning before classification. The combined speech contains a concrete platform proposal and target audience. Return at least one content move (type 1 Generate, 3 Evaluate, 4 Adopt, or 6 Reject); do not return [] merely because types 2 and 5 are absent.\n\nTimeline data:\n${JSON.stringify(partition)}`;
  experiments.push(await callVariant('2. speech_aware_user_message', EVALUATION_SYSTEM_PROMPT, speechAwareUserPrompt));
  const speechAwareSystemPrompt = `${EVALUATION_SYSTEM_PROMPT}\n\nSPECIAL RULE FOR SPEECH-ONLY PARTITIONS:\nWhen adjacent user_speech events form one continuous thought, combine their meaning before classifying. The absence of user_message and ai_response only means types 2 and 5 are absent; it does not mean there are no moves. When the combined speech contains a concrete proposal, idea, target audience, or decision, return at least one content move and do not return [].`;
  experiments.push(await callVariant('3. speech_aware_system_prompt', speechAwareSystemPrompt, buildEvaluationUserPrompt(partition)));
  const combinedSpeechUserPrompt = `Analyze this speech-only design-ideation contribution. Treat the ten chronological fragments as one continuing participant thought.\n\nCombined speech:\n${combinedSpeech}\n\nReturn the applicable content moves only (types 1, 3, 4, or 6) as a JSON array. Do not create Ask or AI_Response moves.\n\nSource timeline events for traceability:\n${JSON.stringify(partition)}`;
  experiments.push(await callVariant('4. combined_speech_context', EVALUATION_SYSTEM_PROMPT, combinedSpeechUserPrompt));
}
console.log(JSON.stringify({
  inspectionIsReadOnly: true,
  fixture: '10 user_speech events from Debug Panel',
  assumedSessionPrecondition: 'manipulation_state exists; this is required in production before either pipeline can run',
  initiationGates: {
    tailHasEvents: partition.length > 0,
    automaticAnalysisRequiresNewSpeech: hasNewSpeech,
    hasInteractionSkeleton: Boolean(hasInteractionSkeleton),
    hasSubstantiveParticipantContent: hasSubstantiveContent,
    runTurnPipelineWouldInitiateEvaluation: runTurnGatePasses,
    runIncrementalPipelineWouldInitiateEvaluation: automaticGatePasses,
    chatSendWouldInitiateEvaluation: runTurnGatePasses,
    noInputSkipReason: !hasInteractionSkeleton && !hasSubstantiveContent
      ? 'No chat/AI-response skeleton and no substantive participant content.'
      : null,
  },
  evaluationAgentContext: {
    expectedAskMoves: 0,
    expectedAIResponseMoves: 0,
    systemPrompt: EVALUATION_SYSTEM_PROMPT,
    userPrompt: buildEvaluationUserPrompt(partition),
    exactTimelinePartitionInput: partition,
  },
  llmExperiments: runExperiments ? experiments : { initiated: false, reason: 'RUN_LLM_EXPERIMENTS is False' },
}, null, 2));
})().catch((error) => { console.error(error); process.exitCode = 1; });
'''

completed = subprocess.run(
    ['node', '-e', inspection_script, str(RUN_LLM_EXPERIMENTS).lower(), json.dumps(TEST_SPEECH)],
    cwd=BACKEND,
    capture_output=True,
    text=True,
    check=False,
)
if completed.returncode != 0:
    raise RuntimeError(completed.stderr or completed.stdout)

inspection = json.loads(completed.stdout)
print(json.dumps(inspection, indent=2, ensure_ascii=False))
